# PA2 End-to-End: SFT + DPO on Kaggle T4 x2

Accelerator: **GPU T4 x2**

This notebook clones the assignment repo, installs deps without replacing PyTorch, then runs:
1. Setup checks
2. Prepare AlpacaEval subset
3. Baseline generation
4. SFT
5. DPO (HH + UltraFeedback)
6. LoRA persona DPO
7. Evaluation generations + pairwise files

Save a notebook version after each major stage.

In [ ]:
import os
REPO_URL = os.environ.get(
    "PA2_REPO_URL",
    "https://github.com/RifaSalman/CS5326.git",
)
REPO_DIR = "/kaggle/working/CS5326/PA2"
CLONE_DIR = "/kaggle/working/CS5326"
STAGE = os.environ.get("PA2_STAGE", "all")  # all|sft|dpo_hh|dpo_uf|lora|generate|judge
print(REPO_URL, STAGE)

In [ ]:
import subprocess, sys
from pathlib import Path

def run(cmd, check=True):
    print("$", " ".join(cmd) if isinstance(cmd, list) else cmd)
    return subprocess.run(cmd, check=check)

if not Path(CLONE_DIR).exists():
    run(["git", "clone", REPO_URL, CLONE_DIR])
else:
    run(["git", "-C", CLONE_DIR, "pull"], check=False)

%cd {REPO_DIR}
run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.43.0,<5", "datasets>=2.18,<5",
    "huggingface_hub", "accelerate", "tqdm", "pyyaml",
    "sentencepiece", "safetensors", "pytest", "matplotlib",
    "peft>=0.11.0", "trl>=0.9.0",
])
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count())
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py

In [ ]:
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)

def should(name):
    return STAGE in {"all", name}

if should("generate") or STAGE == "all":
    !PYTHONPATH=. python scripts/prepare_alpaca_eval.py --output eval_sets/alpaca_eval_300_seed42.json
    !PYTHONPATH=. python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/gen/base_alpaca.json --generator_name qwen2.5-0.5b-base

In [ ]:
if should("sft") or STAGE == "all":
    !PYTHONPATH=. python scripts/train_sft.py --output_dir outputs/sft
    !PYTHONPATH=. python scripts/plot_metrics.py --metrics_path outputs/sft/metrics.jsonl --output_dir report/figures/sft --title SFT

In [ ]:
if should("dpo_hh") or STAGE == "all":
    !PYTHONPATH=. python scripts/train_dpo.py --track hh --sft_checkpoint outputs/sft/best --output_dir outputs/dpo_hh
    !PYTHONPATH=. python scripts/plot_metrics.py --metrics_path outputs/dpo_hh/metrics.jsonl --output_dir report/figures/dpo_hh --title "DPO HH"

if should("dpo_uf") or STAGE == "all":
    !PYTHONPATH=. python scripts/train_dpo.py --track ultrafeedback --sft_checkpoint outputs/sft/best --output_dir outputs/dpo_uf
    !PYTHONPATH=. python scripts/plot_metrics.py --metrics_path outputs/dpo_uf/metrics.jsonl --output_dir report/figures/dpo_uf --title "DPO UltraFeedback"

In [ ]:
if should("lora") or STAGE == "all":
    !PYTHONPATH=. python scripts/train_lora_dpo.py --output_dir outputs/lora_persona_dpo

In [ ]:
if should("generate") or STAGE == "all":
    !PYTHONPATH=. python scripts/generate.py --model_path outputs/sft/best --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/gen/sft_alpaca.json --generator_name sft
    !PYTHONPATH=. python scripts/generate.py --model_path outputs/dpo_uf/best --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/gen/dpo_uf_alpaca.json --generator_name dpo_uf
    !PYTHONPATH=. python scripts/generate.py --model_path outputs/sft/best --prompts_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/gen/sft_hh.json --generator_name sft
    !PYTHONPATH=. python scripts/generate.py --model_path outputs/dpo_hh/best --prompts_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/gen/dpo_hh.json --generator_name dpo_hh
    !PYTHONPATH=. python scripts/build_pairs.py --sft_path outputs/gen/sft_alpaca.json --dpo_path outputs/gen/dpo_uf_alpaca.json --output_path outputs/eval/dpo_uf_vs_sft_pairs.jsonl --sft_name sft --dpo_name dpo_uf
    !PYTHONPATH=. python scripts/build_pairs.py --sft_path outputs/gen/sft_hh.json --dpo_path outputs/gen/dpo_hh.json --output_path outputs/eval/dpo_hh_vs_sft_pairs.jsonl --sft_name sft --dpo_name dpo_hh

In [ ]:
if should("judge") or STAGE == "all":
    !PYTHONPATH=. python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_uf_vs_sft_pairs.jsonl --output_path outputs/eval/dpo_uf_vs_sft_judged.jsonl --reuse_judgments
    !PYTHONPATH=. python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_hh_vs_sft_pairs.jsonl --output_path outputs/eval/dpo_hh_vs_sft_judged.jsonl --reuse_judgments --safety_focus

import shutil
from pathlib import Path
zip_path = Path("/kaggle/working/pa2_outputs.zip")
if Path("outputs").exists():
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", "outputs")
    print("Wrote", zip_path)